# M4 · Cleaning-rule preview (safety gate before facts)

**Goal:** confirm the rules in `config/cleaning_rules.yaml` produce sensible rejection rates *before* we lock them into the fact backfill.

Difference from M2b: M2b counted matches with SQL probes. Here we run the actual `RuleEngine` from `src/accent_fleet/cleaning/rules.py` on a 1-month sample to sanity-check the Python implementation matches the SQL embedded in the fact SQL files.

**Exit criterion:** rejection rates look reasonable (< ~5% for any single rule on any single tenant) or you tune the YAML and re-run.


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs — the rule engine + sample window

In [3]:
from accent_fleet.cleaning.rules import load_rule_engine
engine = load_rule_engine()
print(f"Loaded {len(engine.rules)} rules")
for r in engine.rules:
    targets = ','.join(r.targets)
    print(f"  {r.id:>4} | {r.severity:>8} | {r.action:>8} | {targets:>55} | {r.name}")

SAMPLE_WINDOW = ("2026-03-01", "2026-04-01")
print("Sample window:", SAMPLE_WINDOW)


Loaded 7 rules
    C1 | critical |   reject | path,stop,rep_overspeed,notification,rep_activity_daily | temporal_epoch_error_filter
    C2 | critical |   reject |                                                    path | trip_duration_positive
    C3 |     high |   reject |                                                    path | trip_distance_positive
    C4 | critical |  nullify |                                                    path | fuel_used_sanitize
    C5 |     high |    clamp |                                      path,rep_overspeed | speed_cap
    C6 |   medium |   reject |                                                    stop | stop_duration_bounds
    C7 |     high |   reject |                                                  device | device_vehicle_tenant_chain
Sample window: ('2026-03-01', '2026-04-01')


## 3. Execute

In [4]:
import pandas as pd
import polars as pl
results = []
with get_engine().connect() as conn:
    for tbl in sorted({tbl for r in engine.rules for tbl in r.targets}):
        # pick any available event-time column
        ts_col = {'path':'begin_path_time','rep_overspeed':'begin_path_time','stop':'stop_start','notification':'created_at','rep_activity_daily':'activity_start_time'}.get(tbl)
        if ts_col is None:
            continue
        sample = pd.read_sql(text(f"SELECT * FROM staging.{tbl} WHERE {ts_col} >= :a AND {ts_col} < :b LIMIT 100000"),
                             conn, params={'a': SAMPLE_WINDOW[0], 'b': SAMPLE_WINDOW[1]})
        if sample.empty:
            results.append({'table': tbl, 'rows': 0, 'note': 'no rows in sample window'})
            continue
        pldf = pl.from_pandas(sample)
        cleaned, report = engine.apply(pldf, table=tbl)
        results.append({
            'table': tbl,
            'rows': len(sample),
            'rows_after': len(cleaned),
            'rejected': report.total_rejected,
            'rejected_by_rule': report.rejected_by_rule,
            'clamped_by_rule': report.clamped_by_rule,
            'nullified_by_rule': report.nullified_by_rule,
        })
pd.DataFrame(results)


,table,rows,rows_after,rejected,rejected_by_rule,clamped_by_rule,nullified_by_rule
0,notification,20046,20046,0,{},{},{}
1,path,91665,91635,30,{'C2': 30},{},{'C4': 83}
2,rep_activity_daily,9035,9035,0,{},{},{}
3,rep_overspeed,30528,30528,0,{},{},{}
4,stop,100000,99889,111,{'C6': 111},{},{}


## 4. Inspect — decision gate

In [5]:
import pandas as pd
df = pd.DataFrame(results)
df
# If any single rule rejects >5% of a tenant's data, revisit config/cleaning_rules.yaml.


,table,rows,rows_after,rejected,rejected_by_rule,clamped_by_rule,nullified_by_rule
0,notification,20046,20046,0,{},{},{}
1,path,91665,91635,30,{'C2': 30},{},{'C4': 83}
2,rep_activity_daily,9035,9035,0,{},{},{}
3,rep_overspeed,30528,30528,0,{},{},{}
4,stop,100000,99889,111,{'C6': 111},{},{}
